# FraudGuard: Real-Time Behavioral Credit Card Fraud Detection
### End-to-End Machine Learning Pipeline & Model Benchmark Notebook

This notebook covers the complete ML pipeline for **FraudGuard** matching the presentation deck:
1. **Dataset Loading & Exploration (EDA)**
2. **Feature Engineering & Preprocessing Pipeline (`StandardScaler` + `OneHotEncoder`)**
3. **Handling Extreme Class Imbalance (`scale_pos_weight`)**
4. **Candidate Model Training (Logistic Regression, Random Forest, XGBoost)**
5. **Quantitative Model Performance Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC)**
6. **Precision-Recall Threshold Tuning (F1 Optimization)**
7. **Ultra-Low Latency & SLA Benchmark (< 50ms Target)**
8. **Visualization & Model Export (`final_fraud_model.pkl`)**

## 1. Setup & Environment Imports

In [1]:
import os
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False
    print('Matplotlib/Seaborn not installed. Displaying results in textual formats.')

print('Imports successfully loaded!')

Matplotlib/Seaborn not installed. Displaying results in textual formats.
Imports successfully loaded!


## 2. Dataset Loading & Class Imbalance Exploration

In [2]:
base_dir = Path.cwd()
data_candidates = [
    base_dir / '..' / 'data' / 'fraud_transactions_100k_adjusted_ml_ready.csv',
    base_dir / 'backend' / 'data' / 'fraud_transactions_100k_adjusted_ml_ready.csv',
    base_dir / 'data' / 'fraud_transactions_100k_adjusted_ml_ready.csv'
]

data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Dataset file not found!')

df = pd.read_csv(data_path)
print(f'Loaded dataset: {data_path.name}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

Loaded dataset: fraud_transactions_100k_adjusted_ml_ready.csv
Shape: 100,000 rows x 36 columns


In [3]:
target_col = 'is_fraud' if 'is_fraud' in df.columns else 'label'
y = df[target_col].astype(int)

fraud_count = int(y.sum())
normal_count = len(y) - fraud_count
fraud_ratio = (fraud_count / len(y)) * 100

print(f'Legitimate Transactions : {normal_count:,} ({100 - fraud_ratio:.2f}%)')
print(f'Fraudulent Transactions : {fraud_count:,} ({fraud_ratio:.2f}%)')

if HAS_PLOTS:
    plt.figure(figsize=(6, 3.5))
    sns.barplot(x=['Legitimate (0)', 'Fraudulent (1)'], y=[normal_count, fraud_count], palette=['#10B981', '#EF4444'])
    plt.title('Dataset Class Distribution', fontweight='bold')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

Legitimate Transactions : 98,600 (98.60%)
Fraudulent Transactions : 1,400 (1.40%)


## 3. Preprocessing & Pipeline Construction

In [4]:
ignore_cols = {target_col, 'transaction_id', 'account_id', 'customer_id', 'timestamp', 'created_at', 'date', 'time'}
feature_cols = [c for c in df.columns if c not in ignore_cols]
X = df[feature_cols]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='passthrough'
)
print('Preprocessor ColumnTransformer configured!')

Preprocessor ColumnTransformer configured!


## 4. Stratified Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train split : {len(X_train):,} samples ({y_train.sum():,} frauds)')
print(f'Test split  : {len(X_test):,} samples ({y_test.sum():,} frauds)')

Train split : 80,000 samples (1,120 frauds)
Test split  : 20,000 samples (280 frauds)


## 5. Model Training & Comparison

In [6]:
scale_pos_weight = float(normal_count / max(1, fraud_count))
print(f'Calculated scale_pos_weight = {scale_pos_weight:.2f}')

pipelines = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    'XGBoost (Champion)': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss', n_jobs=-1))
    ])
}

results = {}
for name, pipe in pipelines.items():
    print(f'Training {name}...')
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - t0
    
    y_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.50).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'Accuracy': acc * 100,
        'Precision': prec * 100,
        'Recall': rec * 100,
        'F1-Score': f1 * 100,
        'ROC-AUC': roc_auc * 100,
        'PR-AUC': pr_auc * 100,
        'Fit Time (s)': fit_time,
        'y_proba': y_proba,
        'cm': cm,
        'pipeline': pipe
    }
    print(f'   Completed in {fit_time:.2f}s | Acc: {acc*100:.2f}% | Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1*100:.2f}%')

Calculated scale_pos_weight = 70.43
Training Logistic Regression...
   Completed in 0.70s | Acc: 99.81% | Prec: 89.03% | Rec: 98.57% | F1: 93.56%
Training Random Forest...
   Completed in 2.32s | Acc: 99.97% | Prec: 98.24% | Rec: 99.64% | F1: 98.94%
Training XGBoost (Champion)...
   Completed in 1.23s | Acc: 99.94% | Prec: 96.21% | Rec: 99.64% | F1: 97.89%


## 6. Precision-Recall Threshold Tuning (F1 Maximization)

In [7]:
# Threshold tuning using Precision-Recall Curve
xgb_proba = results['XGBoost (Champion)']['y_proba']
precisions, recalls, thresholds = precision_recall_curve(y_test, xgb_proba)

f1_scores = 2 * (precisions * recalls) / np.maximum(precisions + recalls, 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.50
best_f1 = float(f1_scores[best_idx])

print('=' * 60)
print('DECISION THRESHOLD OPTIMIZATION RESULTS:')
print(f'  Optimal Probability Threshold : {best_threshold:.4f}')
print(f'  Maximized F1-Score            : {best_f1 * 100:.2f}%')
print('=' * 60)

# Threshold Analysis Grid
threshold_grid = [0.10, 0.25, 0.50, 0.75, 0.90]
grid_results = []
for thresh in threshold_grid:
    preds = (xgb_proba >= thresh).astype(int)
    p = precision_score(y_test, preds, zero_division=0) * 100
    r = recall_score(y_test, preds, zero_division=0) * 100
    f = f1_score(y_test, preds, zero_division=0) * 100
    cm = confusion_matrix(y_test, preds)
    grid_results.append({
        'Threshold': thresh,
        'Precision (%)': round(p, 2),
        'Recall (%)': round(r, 2),
        'F1-Score (%)': round(f, 2),
        'False Positives': cm[0, 1],
        'False Negatives': cm[1, 0]
    })

df_thresholds = pd.DataFrame(grid_results)
df_thresholds

DECISION THRESHOLD OPTIMIZATION RESULTS:
  Optimal Probability Threshold : 0.9229
  Maximized F1-Score            : 98.74%


## 7. Ultra-Low Latency & Real-Time SLA Benchmark (< 50ms)

In [8]:
# Measure single-sample inference latency across 1,000 test payloads
xgb_pipe = results['XGBoost (Champion)']['pipeline']
sample_records = X_test.head(1000)

latencies_ms = []
for _, row in sample_records.iterrows():
    payload_df = pd.DataFrame([row])
    t_start = time.perf_counter()
    _ = xgb_pipe.predict_proba(payload_df)
    t_elapsed_ms = (time.perf_counter() - t_start) * 1000.0
    latencies_ms.append(t_elapsed_ms)

avg_latency = np.mean(latencies_ms)
p95_latency = np.percentile(latencies_ms, 95)
p99_latency = np.percentile(latencies_ms, 99)
sla_violations = sum(1 for l in latencies_ms if l > 50.0)
sla_pass_rate = ((len(latencies_ms) - sla_violations) / len(latencies_ms)) * 100

print('=' * 60)
print('REAL-TIME INFERENCE LATENCY & SLA BENCHMARK:')
print(f'  Target SLA Limit          : 50.0 ms')
print(f'  Average Single-Tx Latency : {avg_latency:.2f} ms')
print(f'  95th Percentile (P95)     : {p95_latency:.2f} ms')
print(f'  99th Percentile (P99)     : {p99_latency:.2f} ms')
print(f'  SLA Compliance Pass Rate  : {sla_pass_rate:.2f}% ({sla_violations} violations)')
print('=' * 60)

REAL-TIME INFERENCE LATENCY & SLA BENCHMARK:
  Target SLA Limit          : 50.0 ms
  Average Single-Tx Latency : 8.53 ms
  95th Percentile (P95)     : 11.26 ms
  99th Percentile (P99)     : 15.16 ms
  SLA Compliance Pass Rate  : 99.90% (1 violations)


## 8. Confusion Matrix & Model Export

In [9]:
xgb_cm = results['XGBoost (Champion)']['cm']
tn, fp, fn, tp = xgb_cm.ravel()

print('=' * 50)
print('XGBOOST CHAMPION CONFUSION MATRIX RESULTS:')
print(f'  True Negatives  (Legitimate Approved) : {tn:,}')
print(f'  False Positives (False Alarms)        : {fp}')
print(f'  False Negatives (Missed Fraud)        : {fn}')
print(f'  True Positives  (Fraud Blocked)       : {tp:,}')
print('=' * 50)

output_path = base_dir / 'final_fraud_model.pkl'
if not output_path.parent.exists():
    output_path = base_dir / 'backend' / 'models' / 'final_fraud_model.pkl'

champion_pipeline = results['XGBoost (Champion)']['pipeline']
joblib.dump(champion_pipeline, output_path)
print(f'Successfully saved model pipeline to: {output_path}')

XGBOOST CHAMPION CONFUSION MATRIX RESULTS:
  True Negatives  (Legitimate Approved) : 19,709
  False Positives (False Alarms)        : 11
  False Negatives (Missed Fraud)        : 1
  True Positives  (Fraud Blocked)       : 279
Successfully saved model pipeline to: D:\Download\final npm\fraud detection\backend\final_fraud_model.pkl
